# Treinamento YOLOv8-seg no Google Colab (Projeto FAPESP)
### Aplicação de algoritmos de visão computacional para geração automática de trajetória em selagem robotizada planar

Este notebook executa o treinamento do modelo de **segmentação por instâncias (YOLOv8-seg)** utilizando aceleração por GPU (T4/V100) no Google Colab, com integração direta ao **Roboflow** e salvamento automático dos pesos treinados no **Google Drive**.

## 1. Verificar GPU e Instalar Dependências

In [ ]:
!nvidia-smi
!pip install -q ultralytics roboflow opencv-python matplotlib

## 2. Montar o Google Drive (Para salvar os pesos e relatórios)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Defina o diretório do seu Google Drive onde deseja salvar os pesos treinados
DRIVE_SAVE_DIR = '/content/drive/MyDrive/FAPESP_Modelos'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Diretório no Drive configurado: {DRIVE_SAVE_DIR}")

## 3. Baixar o Dataset de Segmentação do Roboflow
> **Nota:** Insira sua API Key do Roboflow, nome do Workspace e do Projeto abaixo.

In [ ]:
from roboflow import Roboflow

# Substitua com suas credenciais do Roboflow
ROBOFLOW_API_KEY = "SUA_API_KEY_AQUI"
WORKSPACE_NAME = "SEU_WORKSPACE"
PROJECT_NAME = "selagem-planar"
VERSION_NUMBER = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
version = project.version(VERSION_NUMBER)
dataset = version.download("yolov8-seg")

print(f"Dataset baixado com sucesso em: {dataset.location}")

## 4. Iniciar o Treinamento do YOLOv8-seg com GPU

In [ ]:
from ultralytics import YOLO
import os

# Carrega os pesos pré-treinados para segmentação (yolov8n-seg.pt ou yolov8s-seg.pt)
model = YOLO('yolov8n-seg.pt')

# Caminho para o data.yaml gerado pelo Roboflow
yaml_path = os.path.join(dataset.location, 'data.yaml')

# Treinamento
results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    name='fapesp_sealing_yolov8_seg',
    device=0,
    augment=True,      # Data Augmentation ativado conforme proposta
    save=True,
    plots=True
)

## 5. Validar o Modelo e Avaliar Métricas (IoU / F1-Score)

In [ ]:
# Validação no conjunto de validação
metrics = model.val()
print("Métricas de Validação (Mask):")
print(f"mAP50-95 (Mask): {metrics.seg.map:.4f}")
print(f"mAP50 (Mask): {metrics.seg.map50:.4f}")
print(f"mAP75 (Mask): {metrics.seg.map75:.4f}")

## 6. Salvar os Melhores Pesos no Google Drive e Exportar (ONNX / TensorRT)

In [ ]:
import shutil

best_weights_path = 'runs/segment/fapesp_sealing_yolov8_seg/weights/best.pt'
drive_target_pt = os.path.join(DRIVE_SAVE_DIR, 'best_sealing_yolov8n_seg.pt')

if os.path.exists(best_weights_path):
    shutil.copy(best_weights_path, drive_target_pt)
    print(f"Pesos .pt copiados para o Drive: {drive_target_pt}")

# Exporta também para ONNX para inferência otimizada na bancada / Jetson
model.export(format='onnx')
best_onnx_path = 'runs/segment/fapesp_sealing_yolov8_seg/weights/best.onnx'
if os.path.exists(best_onnx_path):
    shutil.copy(best_onnx_path, os.path.join(DRIVE_SAVE_DIR, 'best_sealing_yolov8n_seg.onnx'))
    print("Modelo ONNX salvo no Drive com sucesso!")